In [1]:
import json
import shutil
from pathlib import Path
from time import sleep

import lancedb
import pandas as pd
from loguru import logger

from dreamai.ai import ModelName
from dreamai.rag import application
from dreamai.rag_utils import add_data_with_descriptions
from dreamai.settings import CreatorSettings, RAGAppSettings, RAGSettings
from dreamai.utils import flatten_list

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
creator_settings = CreatorSettings()
rag_settings = RAGSettings()
rag_app_settings = RAGAppSettings()

RERANKER = rag_settings.reranker
HAS_WEB = rag_app_settings.has_web
MODEL = ModelName.GEMINI_FLASH
LANCE_URI = "lance/RFP"
DATA = "/media/hamza/data2/RFP/docs/"

if Path(LANCE_URI).exists():
    shutil.rmtree(LANCE_URI)

lance_db = lancedb.connect(uri=LANCE_URI)

table_descriptions = add_data_with_descriptions(model=MODEL, lance_db=lance_db, data=DATA)

In [ ]:
df = pd.read_csv("RFP.csv")
questions = df.iloc[:, 0].tolist()

In [ ]:
qs = [
    "What Are Your Policies on Price Volatility?",
    "Can You Suggest Relative and Effective Alternatives?",
    "How Long Does Your Company Take to Respond to Issues?",
    "Does the Vendor Own and Service the Products and Equipment It Sells? Or, Is It a Third-Party Broker?",
    "Is Your Company a Registered Woman-, Diverse- or Disabled Veteran-Owned Company?",
    "Why Should Our Company Choose Your Business Over Other Competitors?",
    "Does Your Company Have Any Pending Acquisitions? If So, How Will This Change Your Business Model?",
    "Does Your Company Have Any Legal Issues or Constraints That Could Impact the Performance of Your Products or Services?",
    "Can You Provide a Detailed Implementation Plan — Including a Timeline for the Startup and Transition Process?",
    "How Will You Monitor Progress and Performance on the Account?",
    "How Does Your Company Correct Discrepancies Between Requisition and Items Delivered?",
]

In [3]:
small_rfp = pd.DataFrame(
    {
        "questions": [
            "Are backups performed on a regular basis? Describe backup frequency, retention period and offsite data backup storage",
            "How do you implement and manage JWT access and refresh tokens, including their lifespans and security measures?",
        ]
    }
)
small_rfp.to_csv("/media/hamza/data2/RFP/small_rfp.csv", index=False)

In [4]:
len(
    "icies - Supply chain resilience strategies ## 13 . Cyber Incident Response - Integration with the Cybersecurity Incident Response Plan - Procedures for system isolation and recovery "
)

182

In [ ]:
app = application(db=lance_db, reranker=None, model=MODEL, has_web=False, only_data=True)

In [ ]:
qna = {"questions": questions[-8:-6], "answers": [], "sources": []}

In [ ]:
qna

In [ ]:
for query in qna["questions"]:
    inputs = {"query": query}
    logger.info(f"\nProcessing query: {query}")
    while True:
        step_result = app.step(inputs=inputs)
        if step_result is None:
            logger.error("Error: app.step() returned None")
            break
        action, result, _ = step_result
        logger.info(f"\nAction: {action.name}\n")
        logger.success(f"RESULT: {result}\n")
        if action.name == "terminate":
            break
        elif action.name in ["update_chat_history"]:
            qna["answers"].append(result["chat_history"][-1]["content"].split("\n"))
            qna["sources"].append(
                [
                    {k: v for k, v in json.loads(d).items() if k != "index"}
                    for d in set(
                        [
                            json.dumps(s, sort_keys=True)
                            for s in flatten_list(app.state.get("source_docs", []))
                        ]
                    )
                ]
            )
            sleep(1)
            break